# Import library

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from collections import Counter
import cv2
from glob import glob
from tqdm import tqdm
from termcolor import colored

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\haina\anaconda3\envs\VQA\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\haina\anaconda3\envs\VQA\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\haina\anaconda3\envs\VQA\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\ha

# Custom dataset class

In [5]:
class CustomVOCDataset(torchvision.datasets.VOCDetection):
    def __init__(self, *args, class_mapping, S=7, B=2, C=20, custom_transforms=None, **kwargs):
        super(CustomVOCDataset, self).__init__(*args, **kwargs)
        # initialize YOLO-specific coniguration params
        self.S = S # grid size s x s
        self.B = B # number of bounding boxes
        self.C = C # number of classes
        self.class_mapping = class_mapping # mapping of class names to class indices
        self.custom_transforms = custom_transforms

    def __getitem__(self, index):
        # get an image and its target (annotation) from the VOC dataset
        image, target = super(CustomVOCDataset, self).__getitem__(index)
        img_width, img_height = image.size

        # convert target annotations to YOLO format bounding boxes
        boxes = convert_to_yolo_format(target, img_width, img_height, self.class_mapping)

        just_boxes = boxes[:, 1:]
        labels = boxes[:, 0]

        # transform
        if self.custom_transforms:
            sample = {
                'image': np.array(image),
                'bboxes': just_boxes,
                'labels': labels
            }
            sample = self.custom_transforms(**sample)
            image = sample['image']
            boxes = sample['bboxes']
            labels = sample['labels']

        # create an empty label matrix for YOLO ground truth
        label_matrix = torch.zeros((self.S, self.S, self.C + 5 * self.B))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.float32)
        image = torch.tensor(image, dtype=torch.float32)

        # iterate through each bounding box in YOLO format
        for box, class_label in zip (boxes, labels):
            x, y, width, height = box.tolist()
            class_label = int(class_label)

            # calculate the grid cell (i, j) that this box belongs to
            i, j = int(self.S * y), int(self.S * x)
            x_cell, y_cell = self.S * x - j, self.S * y - i

            # calculate the width and height of the box relative to the grid cell
            width_cell, height_cell = (
                width * self.S,
                height * self.S,
            )

            # if no object has been found in this specific cell (i, j) before:
            if label_matrix[i, j, 20] == 0:
                # mark that an object exists in this cell
                label_matrix[i, j, 20] = 1

                # store the box coordinates as an offset from the cell boundaries:
                box_coordinates = torch.tensor(
                    [x_cell, y_cell, width_cell, height_cell]
                )

                # set the box coordinates in the label matrix
                label_matrix[i, j, 21:25] = box_coordinates

                # set the one-hot encoding for the class label
                label_matrix[i, j, class_label] = 1

        return image, label_matrix

# Other utility functions

In [7]:
def convert_to_yolo_format(target, img_width, img_height, class_mapping):
    """
    convert annotation data from VOC format to YOLO format

    params:
    target (dict): annotation data from VOCDetection dataset
    img_width (int): width of the original image
    img_height (int): height of the original image
    class_mapping (dict): mapping from class names to integer IDs
    
    returns:
    torch.Tensor: tensor of shape [N, 5] for N bounding boxes, each with [class_id, x_center, y_center, width, height]
    """

    # extract the list of annotations from the target dictionary
    annotations = target['annotation']['object']

    # get the real width and height of the image from the annotation
    real_width = int(target['annotation']['size']['width'])
    real_height = int(target['annotation']['size']['height'])

    # ensure the annotations is a list, even if there's only one object
    if not isinstance(annotations, list):
        annotations = [annotations]

    # initialize an empty list to store the converted bounding boxes
    boxes = []

    # loop through each annotation and convert it to YOLO format
    for anno in annotations:
        xmin = int(anno['bndbox']['xmin']) / real_width
        xmax = int(anno['bndbox']['xmax']) / real_width
        ymin = int(anno['bndbox']['ymin']) / real_height
        ymax = int(anno['bndbox']['ymax']) / real_height

        # calcualate the center coordinates, widths and height of the bounding box
        x_center = (xmin + xmax) / 2
        y_center = (ymin + ymax) / 2
        width = xmax - xmin
        height = ymax - ymin

        # retrieve the class name from the annotation and map it to an integer ID
        class_name = anno['name']
        class_id = class_mapping[class_name] if class_name in class_mapping else 0

        # append the YOLO formatted bounding box to the list
        boxes.append([class_id, x_center, y_center, width, height])

    # convert the list of boxes to a torch tensor
    return np.array(boxes)

In [8]:
def intersection_over_union(boxes_preds, boxes_labels, box_format="midpoint"):
    """
    calcualate the intersection over union (IoU) between bounding boxes

    params:
        bboxes_preds (tensor): predicted bounding boxes (BATCH_SIZE, 4)
        boxes_labels (tensor): ground truth bounding boxes (BATCH_SIZE, 4)
        box_format (str): Box format, can be "mid point" or "corners"

    returns:
        tensor: intersection over union scores for each example
    """

    # check if the box format is "mid point"
    if box_format == "midpoint":
        # calculate coordinates of top-left (x1, y1) and bottom-right (x2, y2) points for predicted boxes
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2

        # calculate coordinates of top-left (x1, y1) and bottom-right (x2, y2) points for ground truth boxes
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    # check if the box format is "corners"
    if box_format == "corners":
        # extract coordinates for predicted boxes
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]

        # extract coordinates for ground truth boxes
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    # calculate coordinates of the intersection rectangle
    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # compute the area of the intersection rectangle, clamp(0) to handle cases where they do not overlap
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    # calculate the areas of the predicted and ground truth boxes
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    # calcualte the intersection ober union, adding small epsilon to avoid division by zero
    return intersection / (box1_area + box2_area - intersection + 1e-6)


In [9]:
def non_max_suppresion(bboxes, iou_threshold, threshold, box_format="corners"):
    """
    perform non-maximum supression on a list of bounding boxes
    params:
        bboxes (list): list of bounding boxes, each representedd as [class_pred, probability_score, x1, y1, x2, y2]
        iou_threshold (float): IoU threshold to determine correct predicted bounding boxes
        threshold (float): threshold to discard predicted bounding boxes (independent of IoU)
        box_format (str): "midpoint" or "corners" to specify the format of the bounding boxes

    returns:
        list: list of bounding boxes after performing NMS with a specific IoU threshold
    """

    # check the data type of the input params
    assert type(bboxes) == list

    # Filter predicted bouding boxes based on probality threshold
    bboxes = [box for box in bboxes if box[1] > threshold]

    # sort the bounding boxes by prob in descending order
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)

    # list to store bouding boxes after NMS
    bboxes_after_nms = []

    # continue loop until the list of boudning box is empty
    while bboxes:
        # get the bouding box with the highest prob
        chosen_box = bboxes.pop(0)

        # remove bounding boxes with IoU greater than the specific threshold with the chosen box

        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0]
            or intersection_over_union(
                torch.tensor(chosen_box[2:]),
                torch.tensor(box[2:]),
                box_format=box_format,
            )
            < iou_threshold
        ]

        # append the chosen boundng box on the list
        bboxes_after_nms.append(chosen_box)

    # return the list of bounding box after NMS
    return bboxes_after_nms

In [2]:
def mean_average_precision(
        pred_boxes, true_boxes, iou_threshold=0.5, box_format="midpoint", num_classes=20
):
    """
    calculate the mean average precision (mAP)

    params:
        pred_boxes (list): A list of containing predicted bounding boxes with each box define as [train_idx, class_prediction, prob_score, x1, y2, x2, y2]
        true_boxes (list): similar to pred_boxes but containing information about true boxes
        iou_threshold (float): IoU threshold, where perdicted boxes are cosidered correct
        box_format (Str): "midpoint" or "corners" used to specify the format of the boxes
        num_classes (int): number of classes
    
    returns:
        float: the mAP value across all classes with a specific IoU threshold
    """

    # list to store mAP for each class
    average_precisions = []

    # small epsilon to avoid zero division
    epsilon = 1e-6

    for c in range(num_classes):
        detections = []
        ground_truths = []

        # iterate through all predictions and targets, and only add those belonging to the current class 'c'
        for detection in pred_boxes:
            if detection[1] == c:
                detections.append(detection)

        for true_box in true_boxes:
            if true_box[1] == c:
                ground_truths.append(true_box)

        """
        Find the number of boxes for each training example
        the counter here counts the number of target boxes we have
        for each training example, so if image 0 has 3, image 1 has 5 -> we'll have a dictionary like: amount_bboxes = {0 : 3, 1 : 5}
        """
        amount_bboxes = Counter([gt[0] for gt in ground_truths])

        # we the loop through each key, val in this dictionary and convert it to the following (for the same example):
        # amount_bboxes = {0: torch.tensor([0, 0, 0]), 1: torch.tensor([0, 0, 0, 0, 0])}
        for key, val in amount_bboxes.items():
            amount_bboxes[key] = torch.zeros(val)

        # sort by box probability, index 2 is the probability
        detections.sort(key=lambda x: x[2], reverse=True)
        TP = torch.zeros((len(detections)))
        FP = torch.zeros((len(detections)))

        total_true_bboxes = len(ground_truths)

        # if there are no ground truth boxes for this class, if can be safely skipped 
        if total_true_bboxes == 0:
            continue

        for detection_idx, detection in enumerate(detections):
            # only consider ground truth boxes with the same training index as the prediction
            ground_truth_img = [
                bbox for bbox in ground_truths if bbox[0] == detection[0]
            ]

            num_gts = len(ground_truth_img)
            best_iou = 0
            for idx, gt in enumerate(ground_truth_img):
                iou = intersection_over_union(
                    torch.tensor(detection[3:]),
                    torch.tensor(gt[3:]),
                    box_format=box_format
                )
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            if best_iou > iou_threshold:
                # only detect ground truth once
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    # true position and mark this bounding box as seen
                    TP[detection_idx] = 1
                    amount_bboxes[detection[0]][best_gt_idx] = 1
                else:
                    FP[detection_idx] = 1

            # If IoU is lower, the detection result is false positive
            else:
                FP[detection_idx] = 1

        TP_cumsum = torch.cumsum(TP, dim=0)
        FP_cumsum = torch.cumsum(TP, dim=0)
        recalls = TP_cumsum / (total_true_bboxes + epsilon)
        precisions = torch.divide(TP_cumsum, (TP_cumsum + FP_cumsum + epsilon))
        precisions = torch.cat(torch.tensor([1], precisions))
        recalls = torch.cat((torch.tensor([0]), recalls))
        average_precisions.append(torch.trapz(precisions, recalls))
    return sum(average_precisions) / len(average_precisions)

# draw bounding box on an image
def plot_image(image, boxes):
    """
    draw predicted bounding boxes on an image
    """

    im = np.array(image)
    height, width, _ = im.shape
    
    # create a figure and axis
    fig, ax = plt.subplots(1)

    # display the image
    ax.imshow(im)

    # each box is represented as [x_center, y_center, width, height]
    for box in boxes:
        box = box[2:]
        assert len(box) == 4, "there are more than 4 values in box (x, y, w, h)"
        
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth=1,
            edgecolor="r",
            facecolor="none",
        )
        # add the rectangle to the axis
        ax.add_patch(rect)

    plt.show()

# get predicted and true bounding boxes from the model's output and ground truth data
def get_bboxes(
        loader,
        model,
        iou_threshold,
        threshold,
        pred_format="cell",
        box_format="midpoint",
        device="cuda",
        S=7
):
    all_pred_boxes = []
    all_true_boxes = []

    # ensure model is n evaluation mode before obtaining bounding boxes
    model.eval()
    train_idx = 0
    for batch_idx, (x, label) in enumerate(loader):
        x = x.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            predictions = model(x)

        batch_size = x.shape[0]
        true_bboxes = convert_cellboxes(labels).reshape(batch_size, S * S, -1).tolist()
        bboxes = convert_cellboxes(predictions).reshape(batch_size, S * S, -1).tolist()

        for idx in range(batch_size):
            nms_boxes = non_max_suppresion(
                bboxes[idx],
                iou_threshold=iou_threshold,
                threshold=threshold,
                box_format=box_format,
            )
            for nms_box in nms_boxes:
                all_pred_boxes.append([train_idx] + nms_box)

            for box in true_bboxes[idx]:
                # convert multiple boxes to 0 if predicted
                if box[1] > threshold:
                    all_true_boxes.append([train_idx] + box)
            train_idx += 1

    model.train()
    return all_pred_boxes, all_true_boxes

def get_bboxes_training(
    outputs,
    labels,
    iou_threshold=0.5,
    threshold=0.4,
    box_format="midpoint",
    S=7
):
    all_pred_boxes = []
    all_true_boxes = []
    batch_size = outputs.shape[0]

    # ensure the model is in evaluation mode before obtaining bounding boxes
    train_idx = 0
    true_bboxes = convert_cellboxes(labels).reshape(batch_size, S * S, -1).tolist()

    bboxes = convert_cellboxes(outputs).reshape(batch_size, S * S, -1).tolist()

    for idx in range(batch_size):
        nms_boxes = non_max_suppresion(
            bboxes[idx],
            iou_threshold=iou_threshold,
            threshold=threshold,
            box_format=box_format,
        )

        for nms_box in nms_boxes:
            all_pred_boxes.append([train_idx] + nms_box)

        for box in true_bboxes[idx]:
            # convert multiple boxes to 0 if predicted
            if box[1] > threshold:
                all_true_boxes.append([train_idx] + box)

        train_idx += 1

    return all_pred_boxes, all_true_boxes

def convert_cellboxes(predictions, S=7):
    # convert predictions to CPU for processing
    predictions = predictions.to("cpu")

    # determine the batch size from the predictions shape
    batch_size = predictions.shape[0]

    # reshape predictions to a 3D tensor: [batch size, grid size, grid size, features]
    predictions = predictions.reshape(batch_size, S, S, 30)

    # extract bounding box coordinates for the two boxes predicted for each cell
    bboxes1 = predictions[... , 21:25] # shape [batch size, S, S, 4]
    bboxes2 = predictions[..., 26:30] # shape [batch size, S, S, 4]

    # stack the objectness scores for both boxes and find the box with the higher score
    scores = torch.stack((predictions[..., 20], predictions[..., 25]), dim=-1) # shape: [batch size, S, S, 2]
    best_box = scores.argmax(-1).unsqueeze(-1) # shape [batch size, S, S, 1]

    # use the higher score to select the best bounding box for each cell
    best_boxes = torch.where(best_box == 0, bboxes1, bboxes2) # shape: [batch size, S, S, 4]
    
    # create a grid of cell indices for the x and y coordinates
    cell_indices = torch.arange(S, device=predictions.device).view(1, S, 1).expand(batch_size, S, S).unsqueeze(-1)
    x_indices = cell_indices
    y_indices = cell_indices.permute(0, 2, 1, 3) # permute to align y indices

    # adjust the bounding box coordinates from grid scale to image scale
    x = 1 / S * (best_boxes[..., :1] + x_indices)
    y = 1 / S * (best_boxes[..., 1:2] + y_indices)
    w_h = 1 / S * best_boxes[..., 2:4]

    # concatenate the adjusted bounding box coordinates
    converted_bboxes = torch.cat((x, y, w, h), dim=-1) # shape: [batch size, S, S, 4]

    # determine the class with the highest probability for each cell
    predicted_class = predictions[..., :20].argmax(-1).unsqueeze(-1) # shape [batch size, S, S, 1]

    # find the maximum confidence score for the best bounding box i each cell
    best_confidence = torch.max(predictions[..., 20], predictions[..., 25]).unsqueeze(-1) # shape: [batch size, S, S, 1]

    # concatenate the class predictions, confidence scores, and bounding boxes
    converted_preds = torch.cat((predicted_class, best_confidence, converted_bboxes), dim=-1) # shape [batch size, S, S, 6]

    return converted_preds

# save checkpoint
def save_checkpoint(state, filename="best.pth.tar"):
    print("=> saving... ")
    torch.save(state, filename)

# load checkpoint
def load_checkpoint(checkpoint, model, optimizer):
    print("=> loading... ")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])